# CS 4354 HW3 Part 1: SQL Verification Agent (Colab, standalone)

Everything is inside this notebook, so there is nothing else to upload or clone.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

Steps: check GPU → write the project files → install MySQL and load `soccerhub` → create a read-only DB user → install Ollama + `gpt-oss:20b` → run the agent on Q4, Q6, Q7 → download the logs.

## 1. Check the GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print(r.stdout)

## 2. Write the project files

In [ ]:
import os
for d in ("src", "data", "sql", "logs"):
    os.makedirs(f"/content/hw3/{d}", exist_ok=True)
%cd /content/hw3

In [ ]:
%%writefile /content/hw3/requirements.txt
langchain>=1.0,<2
langchain-ollama>=1.0
langgraph>=1.0
sqlalchemy>=2.0
mysql-connector-python>=9.0
python-dotenv>=1.0


In [ ]:
%%writefile /content/hw3/data/expected.jsonl
{"qid": 1, "prompt": "How many teams are in the database?", "gold_sql": "SELECT COUNT(*) FROM Team;", "gold_value": 6}
{"qid": 2, "prompt": "How many players are currently unattached (teamId is NULL)?", "gold_sql": "SELECT COUNT(*) FROM Player WHERE teamId IS NULL;", "gold_value": 1}
{"qid": 3, "prompt": "How many matches have been scheduled in total?", "gold_sql": "SELECT COUNT(*) FROM `Match`;", "gold_value": 8}
{"qid": 4, "prompt": "How many matches have not yet been played (both scores not recorded)?", "gold_sql": "SELECT COUNT(*) FROM `Match` WHERE homeGoals IS NULL AND awayGoals IS NULL;", "gold_value": 1}
{"qid": 5, "prompt": "How many distinct players have at least one appearance?", "gold_sql": "SELECT COUNT(DISTINCT playerId) FROM Appearance;", "gold_value": 10}
{"qid": 6, "prompt": "How many players have participated in two or more appearances?", "gold_sql": "SELECT COUNT(*) FROM (SELECT playerId FROM Appearance GROUP BY playerId HAVING COUNT(*) >= 2) AS t;", "gold_value": 10}
{"qid": 7, "prompt": "What is the total number of goals scored by Real Madrid across all matches (home plus away)?", "gold_sql": "SELECT SUM(CASE WHEN m.homeTeamId = t.teamId THEN m.homeGoals ELSE m.awayGoals END) FROM `Match` m JOIN Team t ON t.teamId IN (m.homeTeamId, m.awayTeamId) WHERE t.name = 'Real Madrid';", "gold_value": 3}


In [ ]:
%%writefile /content/hw3/sql/soccerhub.sql
DROP DATABASE IF EXISTS soccerhub;
CREATE DATABASE soccerhub CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE soccerhub;

CREATE TABLE Team (
  teamId  INT UNSIGNED PRIMARY KEY AUTO_INCREMENT,
  name    VARCHAR(80) NOT NULL UNIQUE,
  city    VARCHAR(80) NOT NULL,
  stadium VARCHAR(80) NULL
);

CREATE TABLE Player (
  playerId  INT UNSIGNED PRIMARY KEY AUTO_INCREMENT,
  fullName  VARCHAR(100) NOT NULL,
  teamId    INT UNSIGNED NULL,
  position  VARCHAR(20) NULL,
  birthYear SMALLINT NULL,
  FOREIGN KEY (teamId) REFERENCES Team(teamId)
);

CREATE TABLE `Match` (
  matchId    INT UNSIGNED PRIMARY KEY AUTO_INCREMENT,
  homeTeamId INT UNSIGNED NOT NULL,
  awayTeamId INT UNSIGNED NOT NULL,
  matchDate  DATE NOT NULL,
  stadium    VARCHAR(80) NULL,
  homeGoals  TINYINT NULL,
  awayGoals  TINYINT NULL,
  FOREIGN KEY (homeTeamId) REFERENCES Team(teamId),
  FOREIGN KEY (awayTeamId) REFERENCES Team(teamId)
);

CREATE TABLE Appearance (
  matchId  INT UNSIGNED NOT NULL,
  playerId INT UNSIGNED NOT NULL,
  teamId   INT UNSIGNED NOT NULL,
  minutes  SMALLINT NULL,
  goals    TINYINT NULL,
  assists  TINYINT NULL,
  PRIMARY KEY (matchId, playerId),
  FOREIGN KEY (matchId)  REFERENCES `Match`(matchId),
  FOREIGN KEY (playerId) REFERENCES Player(playerId),
  FOREIGN KEY (teamId)   REFERENCES Team(teamId)
);

INSERT INTO Team (teamId, name, city, stadium) VALUES
(1, 'Real Madrid', 'Madrid', 'Santiago Bernabéu'),
(2, 'FC Barcelona', 'Barcelona', 'Olympic Stadium'),
(3, 'Liverpool', 'Liverpool', 'Anfield'),
(4, 'Manchester City', 'Manchester', 'Etihad Stadium'),
(5, 'Paris Saint-Germain', 'Paris', 'Parc des Princes'),
(6, 'AC Milan', 'Milan', 'San Siro');

INSERT INTO Player (playerId, fullName, teamId, position, birthYear) VALUES
(1, 'Vinícius Júnior', 1, 'FW', 2000),
(2, 'Jude Bellingham', 1, 'MF', 2003),
(3, 'Robert Lewandowski', 2, 'FW', 1988),
(4, 'Pedri', 2, 'MF', 2002),
(5, 'Mohamed Salah', 3, 'FW', 1992),
(6, 'Virgil van Dijk', 3, 'DF', 1991),
(7, 'Erling Haaland', 4, 'FW', 2000),
(8, 'Kevin De Bruyne', 4, 'MF', 1991),
(9, 'Kylian Mbappé', 5, 'FW', 1998),
(10, 'Marquinhos', 5, 'DF', 1994),
(11, 'Alisson Becker', 3, 'GK', 1992),
(12, 'Theo Hernández', 6, 'DF', 1997),
(13, 'Paulo Souza', NULL, 'FW', 1999),
(14, 'İlkay Gündoğan', 2, 'MF', 1990);

INSERT INTO `Match` (matchId, homeTeamId, awayTeamId, matchDate, stadium, homeGoals, awayGoals) VALUES
(1, 1, 2, '2025-09-01', 'Santiago Bernabéu', 2, 1),
(2, 3, 5, '2025-09-05', 'Anfield', 0, 0),
(3, 2, 4, '2025-09-10', 'Olympic Stadium', 2, 3),
(4, 1, 5, '2025-09-15', 'Santiago Bernabéu', 1, 2),
(5, 4, 3, '2025-09-20', 'Etihad Stadium', 1, 1),
(6, 5, 2, '2025-09-25', 'Parc des Princes', 2, 0),
(7, 1, 3, '2025-10-02', 'Santiago Bernabéu', 0, 1),
(8, 4, 5, '2025-10-10', 'Etihad Stadium', NULL, NULL);

INSERT INTO Appearance (matchId, playerId, teamId, minutes, goals, assists) VALUES
(1, 1, 1, 90, 1, 0), (1, 2, 1, 88, 1, 1), (1, 3, 2, 90, 1, 0), (1, 4, 2, 90, 0, 1),
(2, 5, 3, 90, 0, 0), (2, 6, 3, 90, 0, 0), (2, 9, 5, 90, 0, 0), (2, 10, 5, 65, 0, NULL),
(3, 3, 2, 90, 1, 0), (3, 4, 2, 85, 1, 1), (3, 7, 4, 90, 2, 0), (3, 8, 4, 88, 1, 1),
(4, 1, 1, 90, 0, 1), (4, 2, 1, 90, 1, 0), (4, 9, 5, 90, 1, 0), (4, 10, 5, 90, 1, 0),
(5, 7, 4, 90, 1, 0), (5, 8, 4, 80, 0, 1), (5, 5, 3, 90, 1, 0), (5, 6, 3, 90, 0, 0),
(6, 9, 5, 90, 1, 1), (6, 10, 5, 90, 1, 0), (6, 3, 2, 90, 0, 0), (6, 4, 2, 78, 0, NULL),
(7, 1, 1, 90, 0, 0), (7, 2, 1, 75, 0, NULL), (7, 5, 3, 90, 1, 0), (7, 6, 3, 90, 0, 0);


In [ ]:
%%writefile /content/hw3/src/db_tools.py
"""
Database tools the agent can call.

    get_schema    - describe the soccerhub tables/columns (plus sample rows)
    execute_sql   - run a READ-ONLY query and return rows or the error
    check_answer  - run a candidate "main" query and say whether its value
                    matches the HW2 ground truth (the expected value itself
                    is never shown to the agent)

Read-only is enforced three ways:
    1. the SQL text is checked before it runs (single SELECT-type statement)
    2. every session is put in READ ONLY transaction mode
    3. the DB account the agent uses (agent_ro) only has SELECT privileges
"""

import json
import os
import re
from decimal import Decimal
from pathlib import Path
from urllib.parse import quote_plus

from dotenv import load_dotenv
from langchain_core.tools import tool
from sqlalchemy import create_engine, inspect, text

ROOT = Path(__file__).resolve().parent.parent
load_dotenv(ROOT / ".env")

DB_USER = os.getenv("DB_USER", "agent_ro")
DB_PASS = os.getenv("DB_PASS", "")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_NAME = os.getenv("DB_NAME", "soccerhub")

EXPECTED_PATH = ROOT / "data" / "expected.jsonl"
MAX_MAIN_ATTEMPTS = 5       # assignment: max 5 tries at the main SQL per question
MAX_ROWS_RETURNED = 25      # keep tool output small so it fits in the context
QUERY_TIMEOUT_MS = 10_000
FLOAT_EPS = 1e-6

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{quote_plus(DB_PASS)}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
)


# ---------------- ground truth ----------------
def _load_expected():
    """qid -> gold value, using the HW2 gold SQL as the source of truth."""
    gold = {}
    with open(EXPECTED_PATH) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                gold[f"Q{row['qid']}"] = row
    return gold


EXPECTED = _load_expected()

# attempts used per question; reset by the runner before each question
attempts = {}


def reset_attempts():
    attempts.clear()


# ---------------- read-only guard ----------------
ALLOWED_START = re.compile(r"^\s*(SELECT|WITH|SHOW|DESCRIBE|DESC|EXPLAIN)\b", re.I)
FORBIDDEN = re.compile(
    r"\b(INSERT|UPDATE|DELETE|REPLACE|MERGE|DROP|ALTER|CREATE|TRUNCATE|RENAME|"
    r"GRANT|REVOKE|LOCK|UNLOCK|CALL|LOAD|HANDLER|SET|COMMIT|ROLLBACK|"
    r"OUTFILE|DUMPFILE|SLEEP|BENCHMARK)\b",
    re.I,
)


def _strip_strings(sql):
    """Remove quoted literals so a word inside a string (e.g. 'Real Madrid')
    can't trip the keyword check."""
    return re.sub(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"", "''", sql)


def check_read_only(sql):
    """Return an error message if the SQL isn't a single read-only statement."""
    s = sql.strip().rstrip(";").strip()
    if not s:
        return "empty query"
    bare = _strip_strings(s)
    if ";" in bare:
        return "only one statement is allowed"
    if not ALLOWED_START.match(bare):
        return "only read-only queries (SELECT / WITH / SHOW / DESCRIBE / EXPLAIN) are allowed"
    bad = FORBIDDEN.search(bare)
    if bad:
        return f"keyword '{bad.group(1).upper()}' is not allowed (read-only access)"
    return None


def _to_jsonable(v):
    if isinstance(v, Decimal):
        return int(v) if v == v.to_integral_value() else float(v)
    if hasattr(v, "isoformat"):
        return v.isoformat()
    return v


def run_query(sql):
    """Run one read-only query. Returns (columns, rows) or raises."""
    problem = check_read_only(sql)
    if problem:
        raise PermissionError(problem)
    with engine.connect() as conn:
        conn.exec_driver_sql("SET SESSION TRANSACTION READ ONLY")
        for stmt in (f"SET SESSION max_execution_time = {QUERY_TIMEOUT_MS}",       # MySQL
                     f"SET SESSION max_statement_time = {QUERY_TIMEOUT_MS / 1000}"):  # MariaDB
            try:
                conn.exec_driver_sql(stmt)
            except Exception:
                pass
        result = conn.execute(text(sql.strip().rstrip(";")))
        cols = list(result.keys())
        rows = [[_to_jsonable(v) for v in r] for r in result.fetchall()]
        conn.rollback()
    return cols, rows


def _same_number(a, b):
    try:
        return abs(float(a) - float(b)) <= FLOAT_EPS
    except (TypeError, ValueError):
        return False


# ---------------- tools ----------------
@tool
def get_schema(table_name: str = "") -> str:
    """Describe the soccerhub database.
    With no table_name: list every table with its columns, types, nullability,
    primary key, foreign keys and row count.
    With a table_name: the same details for that table plus 3 sample rows."""
    insp = inspect(engine)
    tables = insp.get_table_names()
    if table_name:
        match = [t for t in tables if t.lower() == table_name.strip("` ").lower()]
        if not match:
            return json.dumps({"error": f"no table named {table_name}", "tables": tables})
        tables = match

    out = {"database": DB_NAME,
           "note": "Match is a reserved word in MySQL; write the table as `Match`. "
                   "Table names are case-sensitive: use them exactly as listed.",
           "tables": []}
    for t in tables:
        info = {
            "table": t,
            "columns": [
                {"name": c["name"], "type": str(c["type"]), "nullable": c["nullable"]}
                for c in insp.get_columns(t)
            ],
            "primary_key": insp.get_pk_constraint(t).get("constrained_columns", []),
            "foreign_keys": [
                f"{','.join(fk['constrained_columns'])} -> "
                f"{fk['referred_table']}({','.join(fk['referred_columns'])})"
                for fk in insp.get_foreign_keys(t)
            ],
        }
        _, cnt = run_query(f"SELECT COUNT(*) FROM `{t}`")
        info["row_count"] = cnt[0][0]
        if table_name:
            cols, rows = run_query(f"SELECT * FROM `{t}` LIMIT 3")
            info["sample_rows"] = [dict(zip(cols, r)) for r in rows]
        out["tables"].append(info)
    return json.dumps(out, default=str)


@tool
def execute_sql(sql: str) -> str:
    """Run ONE read-only MySQL query (SELECT/WITH/SHOW/DESCRIBE/EXPLAIN) against
    soccerhub and return the columns, up to 25 rows, and the total row count,
    or the error message if the query fails. Use this to explore the data and
    to run independent verification queries. It does NOT count as an attempt."""
    try:
        cols, rows = run_query(sql)
    except Exception as e:
        msg = str(e).splitlines()[0] if str(e) else type(e).__name__
        return json.dumps({"error": msg[:400]})
    return json.dumps({
        "columns": cols,
        "rows": rows[:MAX_ROWS_RETURNED],
        "row_count": len(rows),
        "truncated": len(rows) > MAX_ROWS_RETURNED,
    }, default=str)


@tool
def check_answer(question_id: str, sql: str) -> str:
    """Submit a candidate MAIN query for a question (e.g. question_id="Q6").
    The query is executed and its single numeric result is compared with the
    ground truth. Returns whether it matches, never the expected value.
    Each call uses one of the 5 attempts allowed per question."""
    qid = question_id.strip().upper()
    if not qid.startswith("Q"):
        qid = "Q" + qid
    if qid not in EXPECTED:
        return json.dumps({"error": f"unknown question_id {question_id}",
                           "valid_ids": sorted(EXPECTED)})

    used = attempts.get(qid, 0)
    if used >= MAX_MAIN_ATTEMPTS:
        return json.dumps({
            "error": f"attempt limit reached ({MAX_MAIN_ATTEMPTS}) for {qid}. "
                     "No more checks allowed; if confidence is not established, "
                     "report this question as Unresolved."})
    attempts[qid] = used + 1
    left = MAX_MAIN_ATTEMPTS - attempts[qid]

    try:
        cols, rows = run_query(sql)
    except Exception as e:
        msg = str(e).splitlines()[0] if str(e) else type(e).__name__
        return json.dumps({"attempt": attempts[qid], "attempts_left": left,
                           "matches_ground_truth": False,
                           "error": f"query failed: {msg[:300]}"})

    if len(rows) != 1 or len(cols) != 1:
        return json.dumps({
            "attempt": attempts[qid], "attempts_left": left,
            "matches_ground_truth": False,
            "result_shape": f"{len(rows)} row(s) x {len(cols)} column(s)",
            "note": "the main query must return exactly one row with one numeric value"})

    value = rows[0][0]
    gold_cols, gold_rows = run_query(EXPECTED[qid]["gold_sql"])
    return json.dumps({
        "attempt": attempts[qid],
        "attempts_left": left,
        "result_value": value,
        "matches_ground_truth": _same_number(value, gold_rows[0][0]),
    }, default=str)


ALL_TOOLS = [get_schema, execute_sql, check_answer]


In [ ]:
%%writefile /content/hw3/src/agent.py
"""
CS 4354 - Homework 3 Part 1: an AI agent for SQL query verification.

The agent (LangChain create_agent + gpt-oss:20b on Ollama) gets one
natural-language question at a time and decides on its own which tools to
call: get_schema, execute_sql, check_answer. Python only runs the tools,
enforces read-only access and the 5-attempt limit, and records everything.

Usage:
    python src/agent.py              # run Q4, Q6, Q7
    python src/agent.py Q6           # run just one question

Outputs (in logs/):
    run_<timestamp>.jsonl       every message / tool call / tool result
    run_<timestamp>_report.md   readable transcript + final report per question
    summary_<timestamp>.csv     one row per question
"""

import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from langchain_ollama import ChatOllama

import db_tools
from db_tools import ALL_TOOLS, EXPECTED, MAX_MAIN_ATTEMPTS

ROOT = Path(__file__).resolve().parent.parent
LOG_DIR = ROOT / "logs"

MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:20b")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
NUM_CTX = int(os.getenv("NUM_CTX", "8192"))      # agent conversations get long
REASONING = os.getenv("REASONING", "medium")      # gpt-oss reasoning effort
RECURSION_LIMIT = 60                              # max agent steps per question

QUESTIONS = ["Q4", "Q6", "Q7"]

SYSTEM_PROMPT = f"""You are a careful SQL verification agent working on a MySQL database called soccerhub.

Your job for each question: produce a SQL query that correctly answers the question, and
build real evidence that the query's LOGIC is right, not just its number.

Tools:
- get_schema(table_name): learn the tables, columns, keys and see sample rows.
- execute_sql(sql): run any read-only query. Use it to explore data and to verify.
  It does not count as an attempt.
- check_answer(question_id, sql): submit your candidate MAIN query. It runs the query
  and tells you whether the single value matches the ground truth. You get at most
  {MAX_MAIN_ATTEMPTS} check_answer attempts per question.

Important principles:
- A query can return the expected number while its logic is wrong (for example by
  coincidence, double counting that happens to cancel out, or a filter that happens
  to work on this data). A match from check_answer is necessary but NOT sufficient.
- So, regardless of whether check_answer says it matches, independently investigate
  whether your SQL truly represents the question. Decide yourself what evidence is
  convincing for THIS question. Examples of things you might do (not a required
  checklist): write an alternative query that reaches the answer a different way and
  compare; look at the underlying rows to see what is being counted; check whether a
  JOIN multiplies rows; check that filters (NULL handling, dates, names) capture
  exactly what the question means; check edge cases in the data.
- If you find a problem, revise the query and check again. Base each next step on what
  the previous tool results told you.
- The database is read-only. Never try to modify it.
- MATCH is a reserved word: write the table as `Match` with backticks.
- Table names are case-sensitive on this server: use Team, Player, `Match`, Appearance exactly.

Status rules:
- "Verified" only if check_answer matched AND your independent checks support the logic.
- "Unresolved" if you used all {MAX_MAIN_ATTEMPTS} attempts without that confidence, or if your
  checks contradict the query. Never claim Verified without evidence.

When you are done, reply with ONLY a JSON object (no markdown) in this form:
{{"question_id": "...",
  "final_sql": "...",
  "numerical_answer": <number or null>,
  "matched_ground_truth": true/false,
  "attempts_used": <int>,
  "verification": [{{"check": "what you checked and why", "sql": "...", "result": "...", "outcome": "what it showed"}}],
  "status": "Verified" or "Unresolved",
  "conclusion": "2-4 sentences explaining why you are (or are not) confident"}}
"""


def build_agent():
    llm = ChatOllama(
        model=MODEL,
        base_url=OLLAMA_URL,
        temperature=0,
        num_ctx=NUM_CTX,
        reasoning=REASONING,   # keeps gpt-oss's reasoning separate so we can log it
    )
    return create_agent(model=llm, tools=ALL_TOOLS, system_prompt=SYSTEM_PROMPT)


def parse_report(text):
    """Pull the JSON report out of the agent's final message."""
    if not text:
        return None
    cleaned = re.sub(r"```(?:json)?", "", text).replace("```", "")
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(cleaned[start:end + 1])
    except json.JSONDecodeError:
        return None


def message_events(messages):
    """Turn the agent's message history into a flat list of log events."""
    events = []
    for m in messages:
        if isinstance(m, AIMessage):
            reasoning = (m.additional_kwargs or {}).get("reasoning_content", "")
            if reasoning:
                events.append({"type": "reasoning", "text": reasoning})
            for tc in m.tool_calls or []:
                events.append({"type": "tool_call", "id": tc.get("id"),
                               "tool": tc["name"], "args": tc["args"]})
            if m.content and not m.tool_calls:
                events.append({"type": "final_message", "text": m.content})
        elif isinstance(m, ToolMessage):
            events.append({"type": "tool_result", "id": m.tool_call_id,
                           "tool": m.name, "result": m.content})
    return events


def tool_facts(events):
    """Facts taken from the actual tool results (not from what the agent claims)."""
    checks = [json.loads(e["result"]) for e in events
              if e["type"] == "tool_result" and e["tool"] == "check_answer"]
    checks = [c for c in checks if isinstance(c, dict)]
    return {
        "tool_calls": sum(e["type"] == "tool_call" for e in events),
        "execute_sql_calls": sum(e["type"] == "tool_call" and e["tool"] == "execute_sql"
                                 for e in events),
        "check_answer_calls": len(checks),
        "any_check_matched": any(c.get("matches_ground_truth") for c in checks),
        "last_check_matched": bool(checks and checks[-1].get("matches_ground_truth")),
    }


def run_question(agent, qid):
    db_tools.reset_attempts()
    question = EXPECTED[qid]["prompt"]
    user_msg = (f"question_id: {qid}\nQuestion: {question}\n\n"
                "Produce a correct SQL query for this question and verify it.")
    print(f"\n=== {qid}: {question}")

    t0 = time.time()
    error = ""
    try:
        state = agent.invoke({"messages": [{"role": "user", "content": user_msg}]},
                             config={"recursion_limit": RECURSION_LIMIT})
        messages = state["messages"]
    except Exception as e:      # e.g. recursion limit hit, Ollama down
        error = f"{type(e).__name__}: {str(e).splitlines()[0][:300]}"
        messages = []
    secs = round(time.time() - t0, 1)

    events = message_events(messages)
    final_text = next((e["text"] for e in reversed(events)
                       if e["type"] == "final_message"), "")
    report = parse_report(final_text)
    facts = tool_facts(events)

    # Safety net: the agent can't call something Verified if no check_answer
    # attempt actually matched. (This only checks the claim against the tool
    # log; it doesn't pick queries for the agent.)
    status = (report or {}).get("status", "Unresolved")
    note = ""
    if status == "Verified" and not facts["last_check_matched"]:
        status, note = "Unresolved", "agent said Verified but its last check_answer did not match"
    if error:
        status, note = "Unresolved", f"run stopped: {error}"

    for e in events:
        if e["type"] == "tool_call":
            print(f"  -> {e['tool']}({json.dumps(e['args'])[:150]})")
    print(f"  status: {status}  ({secs}s, {facts['tool_calls']} tool calls)")

    return {"question_id": qid, "question": question, "seconds": secs,
            "events": events, "final_text": final_text, "report": report,
            "facts": facts, "status": status, "status_note": note, "error": error}


# ---------------- writing the logs ----------------
def write_logs(results, stamp):
    LOG_DIR.mkdir(exist_ok=True)

    with open(LOG_DIR / f"run_{stamp}.jsonl", "w") as f:
        for r in results:
            for i, e in enumerate(r["events"], 1):
                f.write(json.dumps({"question_id": r["question_id"], "step": i, **e},
                                   default=str) + "\n")
            f.write(json.dumps({"question_id": r["question_id"], "step": "final",
                                "report": r["report"], "status": r["status"],
                                "status_note": r["status_note"], "facts": r["facts"],
                                "seconds": r["seconds"], "error": r["error"]},
                               default=str) + "\n")

    import csv
    with open(LOG_DIR / f"summary_{stamp}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["question_id", "question", "final_sql", "numerical_answer",
                    "matched_ground_truth", "status", "tool_calls", "check_answer_calls",
                    "execute_sql_calls", "seconds", "note"])
        for r in results:
            rep = r["report"] or {}
            w.writerow([r["question_id"], r["question"], rep.get("final_sql", ""),
                        rep.get("numerical_answer", ""), r["facts"]["last_check_matched"],
                        r["status"], r["facts"]["tool_calls"],
                        r["facts"]["check_answer_calls"], r["facts"]["execute_sql_calls"],
                        r["seconds"], r["status_note"]])

    lines = [f"# Agent execution log ({stamp})", "",
             f"Model: `{MODEL}` (reasoning={REASONING}, num_ctx={NUM_CTX})", "",
             "| Question | Final answer | Matched ground truth | Status | Tool calls | Time |",
             "|---|---|---|---|---|---|"]
    for r in results:
        rep = r["report"] or {}
        lines.append(f"| {r['question_id']} | {rep.get('numerical_answer', '')} | "
                     f"{r['facts']['last_check_matched']} | **{r['status']}** | "
                     f"{r['facts']['tool_calls']} | {r['seconds']}s |")
    for r in results:
        rep = r["report"] or {}
        lines += ["", "---", "", f"## {r['question_id']}: {r['question']}", ""]
        lines += ["### Tool calls", ""]
        n = 0
        for e in r["events"]:
            if e["type"] == "tool_call":
                n += 1
                args = e["args"]
                if "sql" in args:
                    lines += [f"**{n}. `{e['tool']}`**"
                              + (f" ({args.get('question_id')})" if "question_id" in args else ""),
                              "", "```sql", args["sql"], "```"]
                else:
                    lines += [f"**{n}. `{e['tool']}`** `{json.dumps(args)}`"]
            elif e["type"] == "tool_result":
                res = e["result"]
                if e["tool"] == "get_schema" and len(res) > 400:
                    res = res[:400] + " ... (truncated in this view; full text in the .jsonl)"
                lines += ["", "Result:", "```json", res, "```", ""]
        lines += ["### Final report", ""]
        if rep:
            lines += ["- **Final SQL:**", "", "```sql", str(rep.get('final_sql', '')), "```", "",
                      f"- **Numerical answer:** {rep.get('numerical_answer')}",
                      f"- **Matched ground truth (from check_answer):** "
                      f"{r['facts']['last_check_matched']}",
                      f"- **Status:** **{r['status']}**"
                      + (f" ({r['status_note']})" if r["status_note"] else ""),
                      "", "**Independent verification:**", ""]
            for v in rep.get("verification", []) or []:
                if isinstance(v, dict):
                    lines += [f"- **{v.get('check', '')}**", "", "  ```sql",
                              f"  {v.get('sql', '')}", "  ```", "",
                              f"  Result: {v.get('result', '')}. *{v.get('outcome', '')}*", ""]
                else:
                    lines.append(f"- {v}")
            lines += ["", f"**Conclusion:** {rep.get('conclusion', '')}"]
        else:
            lines += [f"Status: **{r['status']}** ({r['status_note'] or 'no JSON report'})",
                      "", "Raw final message:", "", "```", r["final_text"] or "(none)", "```"]
    (LOG_DIR / f"run_{stamp}_report.md").write_text("\n".join(lines) + "\n")


def main():
    qids = [q.upper() for q in sys.argv[1:]] or QUESTIONS
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    print(f"Model {MODEL} | questions {qids}")
    agent = build_agent()
    results = []
    for qid in qids:
        results.append(run_question(agent, qid))
        write_logs(results, stamp)          # save after every question
    print(f"\nLogs written to {LOG_DIR}/ (run_{stamp}.*, summary_{stamp}.csv)")


if __name__ == "__main__":
    main()


## 3. Install MySQL and load the soccerhub database

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y mysql-server zstd > /dev/null
!service mysql start
!sleep 3
!mysql -u root --default-character-set=utf8mb4 < /content/hw3/sql/soccerhub.sql
!mysql -u root -e "SELECT COUNT(*) AS teams FROM soccerhub.Team; SELECT COUNT(*) AS appearances FROM soccerhub.Appearance;"

## 4. Create the agent's read-only database user and the `.env` file
The agent logs in as `agent_ro`, which only has SELECT permission. The password is random and only exists in this session.

In [ ]:
import secrets
ro_pass = secrets.token_hex(12)
!mysql -u root -e "DROP USER IF EXISTS 'agent_ro'@'%'; CREATE USER 'agent_ro'@'%' IDENTIFIED BY '{ro_pass}'; GRANT SELECT, SHOW VIEW ON soccerhub.* TO 'agent_ro'@'%'; FLUSH PRIVILEGES;"

with open("/content/hw3/.env", "w") as f:
    f.write(f"""DB_USER=agent_ro
DB_PASS={ro_pass}
DB_HOST=127.0.0.1
DB_PORT=3306
DB_NAME=soccerhub
OLLAMA_MODEL=gpt-oss:20b
OLLAMA_URL=http://localhost:11434
NUM_CTX=8192
REASONING=medium
""")
print("agent_ro created, .env written")

## 5. Install Python packages

In [ ]:
!pip install -q -r /content/hw3/requirements.txt

## 6. Install Ollama and download gpt-oss:20b (about 13 GB, several minutes)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start. See /content/ollama.log")

In [ ]:
!ollama pull gpt-oss:20b 2>&1 | tail -n 2
!ollama list

## 7. Quick checks before the real run
Expect: a count of 6, then a read-only error for the DELETE, then the model replying, then `ollama ps` showing the model mostly on **GPU**.

In [ ]:
!cd /content/hw3/src && python -c "import db_tools as d; print(d.execute_sql.invoke({'sql': 'SELECT COUNT(*) FROM Team'})); print(d.execute_sql.invoke({'sql': 'DELETE FROM Team'}))"
!ollama run gpt-oss:20b "Reply with just: ready" 2>/dev/null
!ollama ps

## 8. Run the agent on Q4, Q6 and Q7
Each question can take several minutes. Logs are saved after every question.
To rerun one question: `!cd /content/hw3 && python src/agent.py Q6`

In [ ]:
!cd /content/hw3 && python src/agent.py

## 9. Look at the results

In [ ]:
import glob
from IPython.display import Markdown, display
report = sorted(glob.glob("/content/hw3/logs/run_*_report.md"))[-1]
print(report)
display(Markdown(open(report).read()))

## 10. Download the logs

In [ ]:
!cd /content/hw3 && zip -qr /content/hw3_logs.zip logs
from google.colab import files
files.download("/content/hw3_logs.zip")